# Step 1: Environment Setup & Data Preprocessing

This notebook will guide you through verifying your local or Colab Python environment, downloading the Chest X-ray datasets for Tuberculosis, and loading them into memory for processing.

### 1. Verification of Libraries and CUDA (GPU)
Run the following code block to verify that PyTorch is installed correctly and can see your NVIDIA GPU (if available).

In [ ]:
import sys
import torch
import torchvision
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"OpenCV Version: {cv2.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"CUDA Available (GPU Support): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

### 2. Dataset Information

For Tuberculosis detection, we primarily use:
1. **Shenzhen Hospital Dataset**:
   - Contains 662 chest X-rays (326 normal cases, 336 TB cases).
   - Normal images are labeled as `CHNCXR_XXXX_0.png` (ending with `_0.png`).
   - TB images are labeled as `CHNCXR_XXXX_1.png` (ending with `_1.png`).
2. **Montgomery County Dataset**:
   - Contains 138 chest X-rays (80 normal cases, 58 TB cases).
   - Normal images are labeled as `MCUCXR_XXXX_0.png`.
   - TB images are labeled as `MCUCXR_XXXX_1.png`.

#### Downloading the Datasets:
- You can download both datasets directly from Kaggle: [Tuberculosis (TB) Chest X-ray Database](https://www.kaggle.com/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset).
- Download the zip file, extract it, and place the images in the `data/raw/` directory in this workspace:
  - Normal images inside `data/raw/Normal/`
  - TB images inside `data/raw/Tuberculosis/`

### 3. Load Datasets & Build Metadata CSV
Run this block once you have populated the `data/raw/` directories to generate a metadata CSV containing image paths and their corresponding labels (0 for Normal, 1 for Tuberculosis).

In [ ]:
import os
from pathlib import Path

# Define paths
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
NORMAL_DIR = RAW_DIR / "Normal"
TB_DIR = RAW_DIR / "Tuberculosis"

# Check if directories exist
print(f"Data Directory exists: {DATA_DIR.exists()}")
print(f"Normal folder exists: {NORMAL_DIR.exists()}")
print(f"Tuberculosis folder exists: {TB_DIR.exists()}")

In [ ]:
def build_metadata_csv():
    if not NORMAL_DIR.exists() or not TB_DIR.exists():
        print("Please download and place the datasets in data/raw/Normal/ and data/raw/Tuberculosis/ first!")
        return
    
    records = []
    
    # Scan normal cases
    for img_path in NORMAL_DIR.glob("*.png"):
        records.append({"image_path": str(img_path.relative_to(DATA_DIR)), "label": 0, "class_name": "Normal"})
        
    # Scan TB cases
    for img_path in TB_DIR.glob("*.png"):
        records.append({"image_path": str(img_path.relative_to(DATA_DIR)), "label": 1, "class_name": "Tuberculosis"})
        
    df = pd.DataFrame(records)
    
    # Shuffle dataset
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    metadata_file = DATA_DIR / "metadata.csv"
    df.to_csv(metadata_file, index=False)
    print(f"Successfully generated metadata.csv with {len(df)} images at {metadata_file}")
    print(df["class_name"].value_counts())

build_metadata_csv()

### 4. Plotting Sample Images
Once the metadata file is built, run this block to visualize some sample Chest X-rays from both classes.

In [ ]:
metadata_file = DATA_DIR / "metadata.csv"
if metadata_file.exists():
    df = pd.read_csv(metadata_file)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    # Get 4 Normal and 4 TB samples
    normal_samples = df[df["label"] == 0].head(4)
    tb_samples = df[df["label"] == 1].head(4)
    
    for i, (_, row) in enumerate(normal_samples.iterrows()):
        img = cv2.imread(str(DATA_DIR / row['image_path']))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax = axes[0, i]
        ax.imshow(img)
        ax.set_title("Normal")
        ax.axis('off')
        
    for i, (_, row) in enumerate(tb_samples.iterrows()):
        img = cv2.imread(str(DATA_DIR / row['image_path']))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax = axes[1, i]
        ax.imshow(img)
        ax.set_title("Tuberculosis")
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("metadata.csv not found. Please place images in data/ and run Step 3.")